In [1]:
import os
import tensorflow as tf

In [4]:
SRC = '../saved_model/model_gender_final.h5'
DST = '../saved_model/model_gender_dynamic.tflite'

model = tf.keras.models.load_model(SRC)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(DST, "wb") as f:
    f.write(tflite_model)

print("h5: ", round(os.path.getsize(SRC) / 1e6, 2), "MB")
print("tflite: ", round(os.path.getsize(DST) / 1e6, 2), "MB")

INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpq0pcmff5\assets


INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpq0pcmff5\assets


Saved artifact at 'C:\Users\LOQ\AppData\Local\Temp\tmpq0pcmff5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150, 150, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2356960752848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356935755920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356116191568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356116183120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356116191184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356116191376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356117742864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356117743056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356117741328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2356117740560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  235611

In [5]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path="../saved_model/model_gender_dynamic.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input: ", input_details[0]["shape"], input_details[0]["dtype"])
print("Output: ", output_details[0]["shape"], output_details[0]["dtype"])

Input:  [  1 150 150   3] <class 'numpy.float32'>
Output:  [1 2] <class 'numpy.float32'>


d:\Kuliah\Polinema\Semester 5\Machine Learning\faceCount_model\faceCount\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [9]:
import cv2

keras_model = tf.keras.models.load_model('../saved_model/model_gender_final.h5')
interp = tf.lite.Interpreter(model_path="../saved_model/model_gender_dynamic.tflite")
interp.allocate_tensors()
in_idx = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']

folder = '../datasets/Input/'
flips = 0
total = 0

for name in os.listdir(folder):
    if name.endswith('.xml'):
        continue
    img = cv2.imread(os.path.join(folder,name))
    face = cv2.resize(img,(150,150)) / 255.0
    x = np.reshape(face, (1, 150, 150, 3)).astype(np.float32)

    keras_pred = np.argmax(keras_model.predict(x, verbose=0))

    interp.set_tensor(in_idx, x)
    interp.invoke()
    tflite_pred = np.argmax(interp.get_tensor(out_idx))

    total += 1
    if keras_pred != tflite_pred:
        flips += 1
        print(f"Flip: {name} Keras={keras_pred} tflite={tflite_pred}")

print(f"\n{flips}/{total} prediksi berubah ({flips/total*100:.1f}%)")


0/24 prediksi berubah (0.0%)
